In [1]:
#Enter device here, 'cuda' for GPU, and 'cpu' for CPU
device = 'cuda'

import numpy as np 
import matplotlib.pyplot as plt
# from tqdm.notebook import tqdm
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
clean_path = './data/clean/'
noisy_path = './data/noisy/'


In [3]:
clean_path = './data/0001.png'
from skimage import io
clean_im = io.imread(clean_path)
clean_im = clean_im[np.newaxis, np.newaxis, :, :]
clean_im = (clean_im/255.).astype('float32')
clean_img = torch.from_numpy(clean_im)
print(clean_img.shape)
print(clean_img.dtype)

torch.Size([1, 1, 512, 512])
torch.float32


/src/anaconda3/lib/python3.7/site-packages/skimage/io/manage_plugins.py:23: UserWarning: Your installed pillow version is < 7.1.0. Several security issues (CVE-2020-11538, CVE-2020-10379, CVE-2020-10994, CVE-2020-10177) have been fixed in pillow 7.1.0 or higher. We recommend to upgrade this library.
  from .collection import imread_collection_wrapper


In [4]:
class network(nn.Module):
    def __init__(self,n_chan,chan_embed=48):
        super(network, self).__init__()
        
        self.act = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        self.conv1 = nn.Conv2d(n_chan,chan_embed,3,padding=1)
        self.conv2 = nn.Conv2d(chan_embed, chan_embed, 3, padding = 1)
        self.conv3 = nn.Conv2d(chan_embed, n_chan, 1)

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = self.conv3(x)
        
        return x

n_chan = clean_img.shape[1]
model = network(n_chan)
model = model.to(device)
print("The number of parameters of the network is: ",  sum(p.numel() for p in model.parameters() if p.requires_grad))        

The number of parameters of the network is:  21313


In [5]:
def pair_downsampler(img):
    #img has shape B C H W
    c = img.shape[1]
    
    filter1 = torch.FloatTensor([[[[0 ,0.5],[0.5, 0]]]]).to(img.device)
    filter1 = filter1.repeat(c,1, 1, 1)
    
    filter2 = torch.FloatTensor([[[[0.5 ,0],[0, 0.5]]]]).to(img.device)
    filter2 = filter2.repeat(c,1, 1, 1)
    
    output1 = F.conv2d(img, filter1, stride=2, groups=c)
    output2 = F.conv2d(img, filter2, stride=2, groups=c)
    
    return output1, output2

In [6]:
def mse(gt: torch.Tensor, pred:torch.Tensor)-> torch.Tensor:
    loss = torch.nn.MSELoss()
    return loss(gt,pred)

def loss_func(noisy_img):
    noisy1, noisy2 = pair_downsampler(noisy_img)

    pred1 =  noisy1 - model(noisy1)
    pred2 =  noisy2 - model(noisy2)
    
    loss_res = 1/2*(mse(noisy1,pred2)+mse(noisy2,pred1))
    
    noisy_denoised =  noisy_img - model(noisy_img)
    denoised1, denoised2 = pair_downsampler(noisy_denoised)
    
    loss_cons=1/2*(mse(pred1,denoised1) + mse(pred2,denoised2))
    
    loss = loss_res + loss_cons

    return loss

In [7]:
def train(model, optimizer, noisy_img):
  
  loss = loss_func(noisy_img)
  
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  return loss.item()

def test(model, noisy_img, clean_img):
    
    with torch.no_grad():
        pred = torch.clamp(noisy_img - model(noisy_img),0,1)
        MSE = mse(clean_img, pred).item()
        PSNR = 10*np.log10(1/MSE)
    
    return PSNR

def denoise(model, noisy_img):
    
    with torch.no_grad():
        pred = torch.clamp( noisy_img - model(noisy_img),0,1)
    
    return pred 

In [8]:
max_epoch = 1000     # training epochs
lr = 0.001           # learning rate
step_size = 1500     # number of epochs at which learning rate decays
gamma = 0.5          # factor by which learning rate decays

optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

In [15]:
noisy_path = './data/noisy/0001.png'
from skimage import io
noisy_im = io.imread(noisy_path)
noisy_im = noisy_im[np.newaxis, np.newaxis, :, :]
noisy_im = (noisy_im/255.).astype('float32')
noisy_img = torch.from_numpy(noisy_im)
noisy_img = noisy_img.to(device)
clean_img = clean_img.to(device)

In [ ]:
# for epoch in tqdm(range(max_epoch)):

for epoch in range(max_epoch):
    train(model, optimizer, noisy_img)
    scheduler.step()    
    
PSNR = test(model, noisy_img, clean_img)
print(PSNR)

denoised_img = denoise(model, noisy_img)

denoised = denoised_img.cpu().squeeze(0).squeeze(0)
clean = clean_img.cpu().squeeze(0).squeeze(0)
noisy = noisy_img.cpu().squeeze(0).squeeze(0)

denoised = denoised.numpy()
denoised = (denoised * 255.).astype('uint8')
io.imsave('./data/ZSn2n_output/0001.png', denoised)

In [ ]:
for ii in range(1, 401):
    noisy_im = io.imread('./data/noisy/'+str(ii).zfill(4)+'.png')
    noisy_im = noisy_im[np.newaxis, np.newaxis, :, :]
    noisy_im = (noisy_im/255.).astype('float32')
    noisy_img = torch.from_numpy(noisy_im)
    noisy_img = noisy_img.to(device)
    
    clean_img = io.imread('./data/clean/'+str(ii).zfill(4)+'.png')
    clean_img = clean_img[np.newaxis, np.newaxis, :, :]
    clean_img = (clean_img/255.).astype('float32')
    clean_img = torch.from_numpy(clean_img)
    clean_img = clean_img.to(device)
    
    for epoch in range(max_epoch):
        train(model, optimizer, noisy_img)
        scheduler.step()    
    
    PSNR = test(model, noisy_img, clean_img)
    print(ii, PSNR)

    denoised_img = denoise(model, noisy_img)

    denoised = denoised_img.cpu().squeeze(0).squeeze(0)
    denoised = denoised.numpy()
    denoised = (denoised * 255.).astype('uint8')
    io.imsave('./data/ZSn2n_output/'+str(ii).zfill(4)+'.png', denoised)